# **Predicitng ICU Duration from Pre-Surgery Patient Data**

In [101]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, f1_score
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC

### **Loading in the CSV data (downloaded via API)**

In [102]:
df = pd.read_csv("data/hospital_data.csv")

print(df.shape)
df.head()

(6388, 74)


,caseid,subjectid,casestart,caseend,anestart,aneend,opstart,opend,adm,dis,...,intraop_colloid,intraop_ppf,intraop_mdz,intraop_ftn,intraop_rocu,intraop_vecu,intraop_eph,intraop_phe,intraop_epi,intraop_ca
0,1,5955,0,11542,-552,10848.0,1668,10368,-236220,627780,...,0,120,0.0,100,70,0,10,0,0,0
1,2,2487,0,15741,-1039,14921.0,1721,14621,-221160,1506840,...,0,150,0.0,0,100,0,20,0,0,0
2,3,2861,0,4394,-590,4210.0,1090,3010,-218640,40560,...,0,0,0.0,0,50,0,0,0,0,0
3,4,1903,0,20990,-778,20222.0,2522,17822,-201120,576480,...,0,80,0.0,100,100,0,50,0,0,0
4,5,4416,0,21531,-1009,22391.0,2591,20291,-67560,3734040,...,0,0,0.0,0,160,0,10,900,0,2100


In [103]:
features = ["sex", "age", "bmi", "preop_htn", "preop_dm", "asa", "department", "optype", "approach"]
df = df[features + ["icu_days"]]

print(df.shape)
df.head()

(6388, 10)


,sex,age,bmi,preop_htn,preop_dm,asa,department,optype,approach,icu_days
0,M,77.0,26.3,1,0,2.0,General surgery,Colorectal,Open,0
1,M,54.0,19.6,0,0,2.0,General surgery,Stomach,Open,0
2,M,62.0,24.4,0,0,1.0,General surgery,Biliary/Pancreas,Videoscopic,0
3,M,74.0,20.5,1,0,2.0,General surgery,Stomach,Videoscopic,1
4,M,66.0,20.4,1,0,3.0,General surgery,Vascular,Open,13


In [113]:
df['icu_days'].value_counts()

icu_days
0      5084
1       797
2        93
3        78
4        57
5        34
6        29
7        14
8         8
16        7
11        6
38        5
9         5
32        5
14        5
17        4
26        4
12        3
15        2
25        2
21        2
24        2
47        1
19        1
22        1
179       1
13        1
23        1
33        1
42        1
81        1
Name: count, dtype: int64

### **Ensuring data is clean and dropping empty rows**

In [104]:
def icu_bin(x):
    if x == 0:
        return 0
    elif 1 <= x <= 2:
        return 1
    else:
        return 2

df["icu_class"] = df["icu_days"].apply(icu_bin)
target = "icu_class"

In [105]:
df["sex"] = df["sex"].astype(str)
df["department"] = df["department"].astype(str)
df["optype"] = df["optype"].astype(str)
df["approach"] = df["approach"].astype(str)

df["asa"] = pd.to_numeric(df["asa"], errors="coerce")
df["preop_htn"] = pd.to_numeric(df["preop_htn"], errors="coerce")
df["preop_dm"] = pd.to_numeric(df["preop_dm"], errors="coerce")

df = df.dropna()

df.head()

,sex,age,bmi,preop_htn,preop_dm,asa,department,optype,approach,icu_days,icu_class
0,M,77.0,26.3,1,0,2.0,General surgery,Colorectal,Open,0,0
1,M,54.0,19.6,0,0,2.0,General surgery,Stomach,Open,0,0
2,M,62.0,24.4,0,0,1.0,General surgery,Biliary/Pancreas,Videoscopic,0,0
3,M,74.0,20.5,1,0,2.0,General surgery,Stomach,Videoscopic,1,1
4,M,66.0,20.4,1,0,3.0,General surgery,Vascular,Open,13,2


### **75% Train/Test Split**

In [106]:
X = df[features]
y = df[target]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25)

print(X_train.shape, X_test.shape)
print(y_train.shape, y_test.shape)

(4691, 9) (1564, 9)
(4691,) (1564,)


### **Using One-Hot Encoding on Sex (Categorical)**

In [107]:
numeric_features = ["age", "bmi", "preop_htn", "preop_dm", "asa"]
categorical_features = ["sex", "department", "optype", "approach"]

preprocessor = ColumnTransformer(transformers=[
        ("num", "passthrough", numeric_features),
        ("cat", OneHotEncoder(drop="first"), categorical_features)
    ]
)

### **Trying 3 Different Prediciton Models**

In [108]:
models = {
    "Logistic Regression": LogisticRegression(max_iter=500, class_weight="balanced"),
    "Random Forest": RandomForestClassifier(
        n_estimators=400, class_weight="balanced", random_state=42
    ),
    "Gradient Boosting": GradientBoostingClassifier(),
    "SVM (RBF Kernel)": SVC(kernel="rbf", class_weight="balanced")
}

### **Training Each Model**

In [109]:
for name, model in models.items():
    print(name)
    
    classifier = Pipeline(steps=[("prep", preprocessor), ("model", model)])
    classifier.fit(X_train, y_train)
    preds = classifier.predict(X_test)

    print("Accuracy:", accuracy_score(y_test, preds))
    print("Macro F1 Score:", f1_score(y_test, preds, average="macro"))
    print("\nClassification Report:")
    print(classification_report(y_test, preds))
    print("\nConfusion Matrix:")
    print(confusion_matrix(y_test, preds))

Logistic Regression
Accuracy: 0.7391304347826086
Macro F1 Score: 0.5633587026849903

Classification Report:
              precision    recall  f1-score   support

           0       0.95      0.76      0.84      1286
           1       0.40      0.62      0.49       209
           2       0.24      0.74      0.36        69

    accuracy                           0.74      1564
   macro avg       0.53      0.71      0.56      1564
weighted avg       0.85      0.74      0.77      1564


Confusion Matrix:
[[975 182 129]
 [ 42 130  37]
 [  8  10  51]]
Random Forest


/Users/anshbhatnagar/miniconda3/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


Accuracy: 0.850383631713555
Macro F1 Score: 0.62920143145241

Classification Report:
              precision    recall  f1-score   support

           0       0.89      0.95      0.92      1286
           1       0.55      0.37      0.44       209
           2       0.67      0.43      0.53        69

    accuracy                           0.85      1564
   macro avg       0.70      0.59      0.63      1564
weighted avg       0.83      0.85      0.84      1564


Confusion Matrix:
[[1222   54   10]
 [ 126   78    5]
 [  28   11   30]]
Gradient Boosting
Accuracy: 0.8618925831202046
Macro F1 Score: 0.624063227572766

Classification Report:
              precision    recall  f1-score   support

           0       0.89      0.97      0.93      1286
           1       0.63      0.35      0.45       209
           2       0.68      0.39      0.50        69

    accuracy                           0.86      1564
   macro avg       0.73      0.57      0.62      1564
weighted avg       0.84      